# 02 - Preprocessing and Clause Splitting

## Stage 3 Objective

Read `data/processed/extracted_text.csv`, clean each extracted document, split the cleaned text into clause-level records, remove very short clauses, preserve clause numbering where available, and save `data/processed/clauses.csv`.

## Output Columns

- `clause_id`: stable generated id
- `document_id`: source document id from Stage 2
- `source_path`: original file path
- `clause_number`: sequential clause number after filtering
- `clause_label`: detected legal numbering such as `1`, `1.2`, `(a)`, or `Section 4`
- `original_clause_text`: clause text before final flattening
- `clause_text`: cleaned clause text for downstream modeling
- `word_count` and `char_count`: basic size statistics

## 1. Configure Paths and Imports

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.clause_splitter import build_clause_records_for_documents, get_clause_columns
from src.preprocessing import clean_legal_text, count_words

EXTRACTED_TEXT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'extracted_text.csv'
CLAUSES_PATH = PROJECT_ROOT / 'data' / 'processed' / 'clauses.csv'

print(f'Extracted text path: {EXTRACTED_TEXT_PATH}')
print(f'Output path: {CLAUSES_PATH}')

## 2. Load Extracted Text

The notebook expects Stage 2 output. Empty input is handled by writing an empty clauses CSV with the expected schema.

In [ ]:
required_columns = ['document_id', 'source_path', 'page_number', 'text', 'word_count', 'char_count', 'extraction_method', 'error']

if EXTRACTED_TEXT_PATH.exists():
    extracted_df = pd.read_csv(EXTRACTED_TEXT_PATH)
else:
    extracted_df = pd.DataFrame(columns=required_columns)

for column in required_columns:
    if column not in extracted_df.columns:
        extracted_df[column] = ''

print(f'Loaded {len(extracted_df)} extracted row(s).')
display(extracted_df.head())

## 3. Build Document-Level Text

PDF extraction is page-level, so rows are grouped back into document-level text before cleaning and clause splitting.

In [ ]:
working_df = extracted_df.copy()
working_df['text'] = working_df['text'].fillna('').astype(str)
working_df['error'] = working_df['error'].fillna('').astype(str)
working_df['page_number_sort'] = pd.to_numeric(working_df['page_number'], errors='coerce').fillna(10**9)

valid_df = working_df[(working_df['text'].str.strip() != '') & (working_df['error'].str.strip() == '')].copy()

documents = []
if not valid_df.empty:
    for (document_id, source_path), group in valid_df.sort_values('page_number_sort').groupby(['document_id', 'source_path'], dropna=False):
        combined_text = '\n\n'.join(group['text'].tolist())
        cleaned_text = clean_legal_text(combined_text)
        documents.append(
            {
                'document_id': str(document_id),
                'source_path': str(source_path),
                'text': cleaned_text,
                'word_count': count_words(cleaned_text),
                'char_count': len(cleaned_text),
            }
        )

documents_df = pd.DataFrame(documents)
print(f'Prepared {len(documents_df)} document(s) for clause splitting.')
display(documents_df.head())

## 4. Split Into Clauses

Very short clauses are removed with conservative thresholds. Adjust `MIN_CLAUSE_WORDS` and `MIN_CLAUSE_CHARS` if the source documents contain unusually short legal clauses.

In [ ]:
MIN_CLAUSE_WORDS = 8
MIN_CLAUSE_CHARS = 30

records = build_clause_records_for_documents(
    documents,
    min_words=MIN_CLAUSE_WORDS,
    min_chars=MIN_CLAUSE_CHARS,
)
clauses_df = pd.DataFrame(records, columns=get_clause_columns())

print(f'Created {len(clauses_df)} clause row(s).')
display(clauses_df.head(10))

## 5. Preview Clause Numbering and Lengths

In [ ]:
if clauses_df.empty:
    print('No clauses were created. Check that Stage 2 extracted non-empty text rows.')
else:
    preview_df = clauses_df.copy()
    preview_df['clause_preview'] = preview_df['clause_text'].str.slice(0, 220)
    display(
        preview_df[
            ['document_id', 'clause_number', 'clause_label', 'word_count', 'char_count', 'clause_preview']
        ].head(20)
    )
    display(clauses_df.groupby('document_id').size().reset_index(name='clause_count'))

## 6. Save Clauses

In [ ]:
CLAUSES_PATH.parent.mkdir(parents=True, exist_ok=True)
clauses_df.to_csv(CLAUSES_PATH, index=False)
print(f'Saved {len(clauses_df)} clause row(s) to {CLAUSES_PATH.relative_to(PROJECT_ROOT)}')